In [ ]:
import scipy.io as sio
import numpy as np
import torch.nn as nn
import yaml
import torch
import argparse
from tqdm import tqdm
from torch.autograd import Variable
import torch.optim as optim
import matplotlib.pyplot as plt
%matplotlib inline
# define device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
import os
# 注释掉不兼容的 CUDA 内存分配配置
# expandable_segments 选项仅在 PyTorch 1.13+ 版本中支持
# os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
import random
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    # 对于旧版本的 PyTorch，use_deterministic_algorithms 可能不存在
    try:
        torch.use_deterministic_algorithms(True)
    except:
        pass
set_seed(42)

In [ ]:
with open(r'ns.yaml', 'r') as stream:
    config = yaml.load(stream, yaml.FullLoader)
    print(config)

In [ ]:
# load the data
mat_contents = sio.loadmat(r'All_uvp1.mat')
# mat_contents1 = sio.loadmat(r'All_uvp1.mat')
#flag_blue = sio.loadmat(r'1.mat')
#monitor = flag_blue['monitor']

In [ ]:
u = mat_contents['final_u'] # (B, M) (1000, 6724)
v = mat_contents['final_v'] # (B, M) (1000, 6724)
p = mat_contents['final_p']
coor = mat_contents['coors'] # (M, 2) (6724,2)
u = u[:500, :]  # 取前 500 组数据, shape (500, 6724)
v = v[:500, :]  # 取前 500 组数据, shape (500, 6724)
p = p[:500, :]
flag_ID = np.arange(1, 501)

flag_BCxy = mat_contents['flag_BCxy']  # (M, 1)
flag_BCy = mat_contents['flag_BCy']  # (M, 1)
flag_load = mat_contents['flag_BC_load']  # (M, 1)
monitor = mat_contents['monitor']
# monitors = mat_contents1['monitor']

print('raw data shape check (before downsampling):', u.shape, v.shape, coor.shape, flag_BCxy.shape, flag_BCy.shape, flag_load.shape)

# 下采样：减少空间点的数量
downsample_ratio = 1  # 保留50%的点，可以根据需要调整（0.1-1.0之间）
num_points = coor.shape[0]
num_points_downsampled = int(num_points * downsample_ratio)

# 随机选择要保留的点索引（使用固定随机种子以确保可重复性）
np.random.seed(42)
downsample_indices = np.random.choice(num_points, size=num_points_downsampled, replace=False)
downsample_indices = np.sort(downsample_indices)  # 排序以保持空间连续性

# 对数据进行下采样
u = u[:, downsample_indices]  # (500, num_points_downsampled)
v = v[:, downsample_indices]  # (500, num_points_downsampled)
p = p[:, downsample_indices]  # (500, num_points_downsampled)
coor = coor[downsample_indices, :]  # (num_points_downsampled, 2)
flag_BCxy = flag_BCxy[downsample_indices, :]  # (num_points_downsampled, 1)
flag_BCy = flag_BCy[downsample_indices, :]  # (num_points_downsampled, 1)
flag_load = flag_load[downsample_indices, :]  # (num_points_downsampled, 1)
monitor = monitor[downsample_indices, :] if monitor.shape[0] == num_points else monitor

print('downsampled data shape check:', u.shape, v.shape, coor.shape, flag_BCxy.shape, flag_BCy.shape, flag_load.shape)
print(f'Downsampling ratio: {downsample_ratio}, Points: {num_points} -> {num_points_downsampled}')

# 创建一个全为 0 的数组
monitor1 = np.zeros((flag_BCxy.shape[0], 1), dtype=int)

# 随机选择 100 个索引，将其值设置为 1
num_ones = min(100, flag_BCxy.shape[0])  # 设置值为 1 的数量，不超过总点数
indices = np.random.choice(flag_BCxy.shape[0], size=num_ones, replace=False)
monitor1[indices] = 1
flag_bc = np.copy(flag_BCxy)
# flag_load =  (flag_load == 1) | (flag_BCy == 1) | (flag_BCxy == 1)| (flag_BCxy == 0)
flag_bc =   (monitor1 == 1) | (flag_load == 1) | (flag_BCy == 1)

num_nei = coor[np.where(flag_BCxy + flag_BCy + flag_load == 0)[0], :]
# 打印点的数量
print(f'Number of points where flag_BCxy + flag_BCy + flag_load == 0: {len(num_nei)}')

num_flag_BCxy = coor[np.where(flag_BCxy == 1)[0], :]
# 打印点的数量
print(f'Number of points where flag_BCxy == 1: {len(num_flag_BCxy)}')

coors_iflag_BCy = coor[np.where(flag_BCy == 1)[0], :]
# 打印点的数量
print(f'Number of points where flag_BCy == 1: {len(coors_iflag_BCy)}')

coors_flag_load = coor[np.where(flag_load == 1)[0], :]
# 打印点的数量
print(f'Number of points where flag_load == 1: {len(coors_flag_load)}')


coors_flag_bc = coor[np.where(flag_bc == 1)[0], :]
# 打印点的数量
print(f'Number of points where flag_bc == 1: {len(coors_flag_bc)}')



In [ ]:
# load the scalar factor加载标量因子
#scalar_factor = 1e-4  #标量因子
dens = mat_contents['dens'][0][0]#密度
mu = mat_contents['mu'][0][0]  #动态粘度

# 构造参数输入 
id_param = np.where(flag_bc==1)[0]
datasize = u.shape[0]
w=u[:,id_param]
# params = np.concatenate((np.repeat(np.expand_dims(coor[id_param,:],0),datasize,axis=0),np.expand_dims(u[:,id_param],-1),np.expand_dims(v[:,id_param],-1), np.expand_dims(p[:,id_param],-1)), -1)  
params_u = np.concatenate((np.repeat(np.expand_dims(coor[id_param,:],0),datasize,axis=0),np.expand_dims(u[:,id_param],-1)), -1) 
params_v = np.concatenate((np.repeat(np.expand_dims(coor[id_param,:],0),datasize,axis=0),np.expand_dims(v[:,id_param],-1)), -1) 
params_p = np.concatenate((np.repeat(np.expand_dims(coor[id_param,:],0),datasize,axis=0),np.expand_dims(p[:,id_param],-1)), -1) 
# num_bc_nodes = params.shape[1]

# define dataset as torch.tensor 将数据集定义为torch.tensor
params_u = torch.tensor(params_u)    # (B, N, 3)
params_v = torch.tensor(params_v) 
params_p = torch.tensor(params_p) 
u = torch.tensor(u)    # (B, M)
v = torch.tensor(v)    # (B, N)
p = torch.tensor(p) 
flag_ID = torch.tensor(flag_ID)
coors = torch.tensor(coor)    # (M,2)
print(params_u.shape)
print(params_v.shape)
print(params_p.shape)

print(u.shape)
print(v.shape)
print(p.shape)

In [ ]:
# define data loader
# bar1 = [0,int(0.7*datasize)]
# bar2 = [int(0.7*datasize),int(0.8*datasize)]
# bar3 = [int(0.8*datasize),int(datasize)]
# 生成随机索引
torch.manual_seed(42)
indices = torch.randperm(datasize)  # 随机生成 [0, datasize) 的打乱索引

# 根据随机索引重新排列数据
params_u = params_u[indices]
params_v = params_v[indices]
params_p = params_p[indices]
u = u[indices]
v = v[indices]
p = p[indices]
flag_ID = flag_ID[indices]
# 划分数据集
bar1 = [0, int(0.7 * datasize)]  # 前 70% 为训练集
bar2 = [int(0.7 * datasize), int(0.8 * datasize)]  # 10% 为验证集
bar3 = [int(0.8 * datasize), datasize]  # 剩余 20% 为测试集

train_dataset = torch.utils.data.TensorDataset(params_u[bar1[0]:bar1[1],:,:],params_v[bar1[0]:bar1[1],:,:],params_p[bar1[0]:bar1[1],:,:], u[bar1[0]:bar1[1],:], v[bar1[0]:bar1[1],:], p[bar1[0]:bar1[1],:])
val_dataset = torch.utils.data.TensorDataset(params_u[bar2[0]:bar2[1],:,:],params_v[bar2[0]:bar2[1],:,:], params_p[bar2[0]:bar2[1],:,:], u[bar2[0]:bar2[1],:], v[bar2[0]:bar2[1],:], p[bar2[0]:bar2[1],:])
test_dataset = torch.utils.data.TensorDataset(params_u[bar3[0]:bar3[1],:,:], params_v[bar3[0]:bar3[1],:,:], params_p[bar3[0]:bar3[1],:,:], u[bar3[0]:bar3[1],:], v[bar3[0]:bar3[1],:], p[bar3[0]:bar3[1],:],flag_ID[bar3[0]:bar3[1]])


train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=config['train']['batchsize'], shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=config['train']['batchsize'], shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=2, shuffle=False)

In [ ]:
class fc(nn.Module):

    def __init__(self, dim):
        super().__init__()
        self.ff = nn.Linear(dim, dim)
        self.act = nn.Tanh()

    def forward(self, x):
        return self.act(self.ff(x))

In [ ]:
class DeepMONet(nn.Module):
    def __init__(self, config):
        """
        mode: 'u', 'v', 'p' 分别表示不同的模型
        """
        mode_u = 'u'  # 对应模式u
        mode_v = 'v'  # 对应模式v
        mode_p = 'p'  # 对应模式p
        super().__init__()
        self.mode_u = mode_u
        self.mode_v = mode_v
        self.mode_p = mode_p
        # branch network 1
        trunk_layers_u = [nn.Linear(3, config['model']['fc_dim']), nn.Tanh()]
        for _ in range(config['model']['N_layer'] - 1):
            trunk_layers_u.append(nn.Linear(config['model']['fc_dim'], config['model']['fc_dim']))
            trunk_layers_u.append(nn.Tanh())
        trunk_layers_u.append(nn.Linear(config['model']['fc_dim'], config['model']['fc_dim']))
        self.branch_u = nn.Sequential(*trunk_layers_u)

        trunk_layers_v = [nn.Linear(3, config['model']['fc_dim']), nn.Tanh()]
        for _ in range(config['model']['N_layer'] - 1):
            trunk_layers_v.append(nn.Linear(config['model']['fc_dim'], config['model']['fc_dim']))
            trunk_layers_v.append(nn.Tanh())
        trunk_layers_v.append(nn.Linear(config['model']['fc_dim'], config['model']['fc_dim']))
        self.branch_v = nn.Sequential(*trunk_layers_v)

        trunk_layers_p = [nn.Linear(3, config['model']['fc_dim']), nn.Tanh()]
        for _ in range(config['model']['N_layer'] - 1):
            trunk_layers_p.append(nn.Linear(config['model']['fc_dim'], config['model']['fc_dim']))
            trunk_layers_p.append(nn.Tanh())
        trunk_layers_p.append(nn.Linear(config['model']['fc_dim'], config['model']['fc_dim']))
        self.branch_p = nn.Sequential(*trunk_layers_p)

         # trunk network  
        def fc_layers(fc_dim):
            layers = [nn.Linear(fc_dim, fc_dim)]
            return nn.Sequential(*layers)
        self.FC1 = nn.Linear(2, config['model']['fc_dim'])
        self.trunk_layers = nn.ModuleList(
            [nn.ModuleList([fc_layers(config['model']['fc_dim'])
                            for _ in range(config['model'][f'N_layer_{mode}'])]) 
             for mode in ['u', 'v', 'p']]
        )

        self.act = nn.Tanh()
    
    def forward(self, x_coor, y_coor, par_u, par_v, par_p):
        
        xy = torch.cat((x_coor.unsqueeze(-1), y_coor.unsqueeze(-1)), -1)  # (B, M, 2)
        enc_u = self.branch_u(par_u)
        enc_u = torch.amax(enc_u, 1, keepdim=True)
        u = self.FC1(xy)
        u = self.act(u)
          
        enc_v = self.branch_v(par_v)
        enc_v = torch.amax(enc_v, 1, keepdim=True)
        v = self.FC1(xy)
        v = self.act(v)
          
        enc_p = self.branch_p(par_p)
        enc_p = torch.amax(enc_p, 1, keepdim=True)
        p = self.FC1(xy)
        p = self.act(p)
        if self.mode_u == 'u':  
            for i, trunk in enumerate(self.trunk_layers[0]):
                u = u * enc_u
                u = trunk(u)
                if i < len(self.trunk_layers[0]) - 1:  
                    u = self.act(u)      
            u = torch.mean(u * enc_u, -1)  # (B, M)

        if self.mode_v == 'v':
            for i, trunk in enumerate(self.trunk_layers[1]):
                v = v * enc_v
                v = trunk(v)
                if i < len(self.trunk_layers[1]) - 1:  
                    v = self.act(v)      
            v = torch.mean(v * enc_v, -1)  # (B, M)

        if self.mode_p == 'p':
            for i, trunk in enumerate(self.trunk_layers[2]):
                p = p * enc_p
                p = trunk(p)
                if i < len(self.trunk_layers[2]) - 1:  
                    p = self.act(p)      
            p = torch.mean(p * enc_p, -1)  # (B, M)
        return u, v, p
          
  

In [ ]:

def val(model, loader, coors, device):

    # split the coordinates
    test_coor_x = coors[:, 0].unsqueeze(0).float().to(device)
    test_coor_y = coors[:, 1].unsqueeze(0).float().to(device)

    mean_relative_L2 = 0
    mean_relative_L2_u = 0
    mean_relative_L2_v = 0
    mean_relative_L2_p = 0
    num = 0
    num_u = 0
    num_v = 0
    num_p = 0

    for (par_u, par_v, par_p, u, v, p) in loader:
        
        # move the data to device
        batch = par_u.shape[0]

        par_u = par_u.float().to(device)
        par_v = par_v.float().to(device)
        par_p = par_p.float().to(device)
        u = u.float().to(device)
        v = v.float().to(device)
        p = p.float().to(device)
        # model forward
        u_pred, v_pred, p_pred = model(test_coor_x.repeat(batch,1), test_coor_y.repeat(batch,1), par_u, par_v, par_p)
        
        
        L2_relative = torch.sqrt(torch.sum((u_pred-u)**2 + (v_pred-v)**2 + (p_pred-p)**2, -1)) / torch.sqrt(torch.sum((u)**2+ (v)**2+ (p)**2 , -1))
        L2_relative_u = torch.sqrt(torch.sum((u_pred-u)**2 , -1)) / torch.sqrt(torch.sum((u)**2 , -1))
        L2_relative_v = torch.sqrt(torch.sum((v_pred-v)**2 , -1)) / torch.sqrt(torch.sum((v)**2 , -1))
        L2_relative_p = torch.sqrt(torch.sum((p_pred-p)**2 , -1)) / torch.sqrt(torch.sum((p)**2 , -1))

        # compute relative error
        mean_relative_L2 += torch.sum(L2_relative)
        num += u.shape[0] 
        
        mean_relative_L2_u += torch.sum(L2_relative_u)
        num_u += u.shape[0]
        mean_relative_L2_v += torch.sum(L2_relative_v)
        num_v += v.shape[0]
        mean_relative_L2_p += torch.sum(L2_relative_p)
        num_p += p.shape[0]

        # compute absolute error for point sampling probability computation
        # abs_err = torch.mean(torch.abs(u_pred-u) + torch.abs(v_pred-v)  + torch.abs(p_pred-p), 0).detach().cpu().numpy()
        
        # # check GPU usage
        # print("torch.cuda.memory_allocated: %fGB"%(torch.cuda.memory_allocated(0)/1024/1024/1024))
    mean_relative_L2 /= num
    mean_relative_L2 = mean_relative_L2.detach().cpu().item()
    mean_relative_L2_u /= num_u
    mean_relative_L2_u = mean_relative_L2_u.detach().cpu().item()
    mean_relative_L2_v /= num_v
    mean_relative_L2_v = mean_relative_L2_v.detach().cpu().item()
    mean_relative_L2_p /= num_p
    mean_relative_L2_p = mean_relative_L2_p.detach().cpu().item()

    return mean_relative_L2_u, mean_relative_L2_v, mean_relative_L2_p, mean_relative_L2

In [ ]:
import csv

def save_relative_errors(errors, epochs, filename, config, args):
    # 尝试读取现有的文件，如果文件不存在，则初始化为空
    try:
        with open(filename, 'r', newline='') as f:
            reader = csv.reader(f)
            # 跳过标题
            next(reader)
            # 获取 Epochs 列的所有值
            existing_epochs = [int(row[0]) for row in reader if row]  # 只读取非空行
    except FileNotFoundError:
        # 如果文件不存在，初始化 existing_epochs 为空列表
        existing_epochs = []

    visual_freq = config['train']['visual_freq']
    # 确保追加的 Epochs 是从已有的最大值开始
    if existing_epochs:
        last_epoch = existing_epochs[-1]  # 获取文件中最后一个 Epoch 值
        n =  config['train']['visual_freq']
    else:
        last_epoch = config['train']['visual_freq']  # 如果文件为空或不存在
        n = 0

    # 生成新的 Epochs，根据传入的 epochs 递增
    new_epochs = [last_epoch + (i + n) for i in epochs]   
    
    # 将数据追加到文件
    with open(filename, 'a', newline='') as f:
        writer = csv.writer(f)

        # 如果文件为空，写入标题行
        if not existing_epochs:
            writer.writerow(['Epoch', 'err_u', 'err_v', 'err_p', 'err'])  # 写入标题行
        
        # 在写入新数据之前，先写入一个空行
        if existing_epochs:
            writer.writerow([])  # 空行
        # 写入新的 Epochs 和相应的 Relative_Error 数据
        for epoch, (err_u, err_v, err_p, err) in zip(new_epochs, errors):
            writer.writerow([epoch, err_u, err_v, err_p, err])  # 写入新数据
            


In [ ]:
def plot_relative_errors_from_file(file_path):
    # 读取文件数据，假设文件格式是: Epochs, Relative_Error
    data = np.loadtxt(file_path, delimiter=',', skiprows=1)  # 跳过第一行标题
    epochs = data[:, 0]  # 第1列为 epochs
    err_u = data[:, 1]  # 第2列为 err_u
    err_v = data[:, 2]  # 第3列为 err_v
    err_p = data[:, 3]  # 第4列为 err_p
    err = data[:, 4]    # 第5列为 err

    # 绘制相对误差与 epochs 的关系
    plt.figure(figsize=(12, 8))
    plt.plot(epochs, err_u, marker='o', label="err_u", color='r')
    plt.plot(epochs, err_v, marker='s', label="err_v", color='g')
    plt.plot(epochs, err_p, marker='^', label="err_p", color='b')
    plt.plot(epochs, err, marker='x', label="err", color='m')
    
    plt.title("L2 Relative Errors vs Epochs", fontsize=14)
    plt.xlabel("Epochs", fontsize=12)
    plt.ylabel("L2 Relative Errors", fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(fontsize=12)

    # 保存并展示图片
    plt.savefig('relative_error_plot_from_file.png')
    plt.show()
    print("Plot has been saved and displayed.")

In [ ]:
# function for testing
def test(model, loader, coors, device, args):
    '''
    Input:
        model: the model instance to be tested
        loader: testing loader of the dataset
        coors: A set of fixed coordinate
        device: cpu or gpu
        args: usig this information to assign name for the output plots
    Ouput:
        A plot of the PDE solution ground-truth, prediction, and absolute error
    '''

    # split the coordinates
    test_coor_x = coors[:, 0].unsqueeze(0).float().to(device)
    test_coor_y = coors[:, 1].unsqueeze(0).float().to(device)

    mean_relative_L2 = 0
    num = 0
    max_relative_err = -1
    min_relative_err = np.inf

    mean_relative_L2_u = 0
    num_u = 0
    max_relative_err_u = -1
    min_relative_err_u = np.inf

    mean_relative_L2_v = 0
    num_v = 0
    max_relative_err_v = -1
    min_relative_err_v = np.inf

    mean_relative_L2_p = 0
    num_p = 0
    max_relative_err_p = -1
    min_relative_err_p = np.inf
    
    target_id = args.target_id  # 你想查找的 flag_ID
    found_u, found_v, found_p = None, None, None  

    for (par_u, par_v, par_p, u, v, p, flag_ID) in loader:

        # move the data to device
        batch = par_u.shape[0]
        par_u = par_u.float().to(device)
        u = u.float().to(device)
        flag_ID = flag_ID.float().to(device)
        
        par_v = par_v.float().to(device)
        v = v.float().to(device)
        
        par_p = par_p.float().to(device)
        p = p.float().to(device)

        # model forward
        u_pred, v_pred, p_pred = model(test_coor_x.repeat(batch,1), test_coor_y.repeat(batch,1), par_u, par_v, par_p)
        
        
        L2_relative = torch.sqrt(torch.sum((u_pred-u)**2 + (v_pred-v)**2 + (p_pred-p)**2, -1)) / torch.sqrt(torch.sum((u)**2 + (v)**2 + (p)**2, -1))
        L2_relative_u = torch.sqrt(torch.sum((u_pred-u)**2 , -1)) / torch.sqrt(torch.sum((u)**2 , -1))
        L2_relative_v = torch.sqrt(torch.sum((v_pred-v)**2 , -1)) / torch.sqrt(torch.sum((v)**2 , -1))
        L2_relative_p = torch.sqrt(torch.sum((p_pred-p)**2, -1)) / torch.sqrt(torch.sum((p)**2, -1))
        
        max_err, max_err_idx = torch.topk(L2_relative, 1)
        max_err_u, max_err_idx_u = torch.topk(L2_relative_u, 1)
        max_err_v, max_err_idx_v = torch.topk(L2_relative_v, 1)
        max_err_p, max_err_idx_p = torch.topk(L2_relative_p, 1)

        if max_err > max_relative_err:
            max_relative_err = max_err
        min_err, min_err_idx = torch.topk(-L2_relative, 1)
        min_err = -min_err
        if min_err < min_relative_err:
            min_relative_err = min_err
        mean_relative_L2 += torch.sum(L2_relative).detach().cpu().item()
        num += u.shape[0]


        #查找想到的ID
        if int(flag_ID[max_err_idx_u].detach().cpu().numpy().item()) == int(target_id):
            found_ID = int(flag_ID[max_err_idx_u].detach().cpu().numpy().item())
            found_u = u[max_err_idx_u].detach().cpu().numpy()  # 转换为 NumPy
            found_v = v[max_err_idx_u].detach().cpu().numpy()
            found_p = p[max_err_idx_u].detach().cpu().numpy()
            found_u_pred = u_pred[max_err_idx_u].detach().cpu().numpy()  # 转换为 NumPy
            found_v_pred = v_pred[max_err_idx_u].detach().cpu().numpy()
            found_p_pred = p_pred[max_err_idx_u].detach().cpu().numpy()
            print(f"找到样本 ID: {target_id}")
                
        
        if max_err_u > max_relative_err_u:
             max_relative_err_u = max_err_u
             worst_flag_ID_u = flag_ID[max_err_idx_u].detach().cpu().numpy()
             worst_f_u = u_pred[max_err_idx_u,:].detach().cpu().numpy()
             worst_gt_u = u[max_err_idx_u,:].detach().cpu().numpy()
        min_err_u, min_err_idx_u = torch.topk(-L2_relative_u, 1)
        min_err_u = -min_err_u
        if min_err_u < min_relative_err_u:
             min_relative_err_u = min_err_u
             best_flag_ID_u = flag_ID[min_err_idx_u].detach().cpu().numpy()
             best_f_u = u_pred[min_err_idx_u,:].detach().cpu().numpy()
             best_gt_u = u[min_err_idx_u,:].detach().cpu().numpy()
        mean_relative_L2_u += torch.sum(L2_relative_u).detach().cpu().item()
        num_u += u.shape[0]

        if max_err_v > max_relative_err_v:
            max_relative_err_v = max_err_v
            worst_flag_ID_v = flag_ID[max_err_idx_v].detach().cpu().numpy()
            worst_f_v = v_pred[max_err_idx_v,:].detach().cpu().numpy()
            worst_gt_v = v[max_err_idx_v,:].detach().cpu().numpy()
        min_err_v, min_err_idx_v = torch.topk(-L2_relative_v, 1)
        min_err_v = -min_err_v
        if min_err_v < min_relative_err_v:
            min_relative_err_v = min_err_v
            best_flag_ID_v = flag_ID[min_err_idx_v].detach().cpu().numpy()
            best_f_v = v_pred[min_err_idx_v,:].detach().cpu().numpy()
            best_gt_v = v[min_err_idx_v,:].detach().cpu().numpy()
        mean_relative_L2_v += torch.sum(L2_relative_v).detach().cpu().item()
        num_v += v.shape[0]
            
        if max_err_p > max_relative_err_p:
            max_relative_err_p = max_err_p
            worst_flag_ID_p = flag_ID[max_err_idx_p].detach().cpu().numpy()
            worst_f_p = p_pred[max_err_idx_p,:].detach().cpu().numpy()
            worst_gt_p = p[max_err_idx_p,:].detach().cpu().numpy()
        min_err_p, min_err_idx_p = torch.topk(-L2_relative_p, 1)
        min_err_p = -min_err_p
        if min_err_p < min_relative_err_p:
            min_relative_err_p = min_err_p
            best_flag_ID_p = flag_ID[min_err_idx_p].detach().cpu().numpy()
            best_f_p = p_pred[min_err_idx_p,:].detach().cpu().numpy()
            best_gt_p = p[min_err_idx_p,:].detach().cpu().numpy()
        mean_relative_L2_p += torch.sum(L2_relative_p).detach().cpu().item()
        num_p += p.shape[0]

        one_flag_ID = int(flag_ID[max_err_idx_u].detach().cpu().numpy().item())
        one_relative_L2_u = torch.sum(L2_relative_u).detach().cpu().item()
        one_num_u = u.shape[0]
        one_relative_L2_v = torch.sum(L2_relative_v).detach().cpu().item()
        one_num_v = v.shape[0]
        one_relative_L2_p = torch.sum(L2_relative_p).detach().cpu().item()
        one_num_p = p.shape[0]
        one_relative_L2_u /= one_num_u
        one_relative_L2_v /= one_num_v
        one_relative_L2_p /= one_num_p
        
        file_path = "relative_L2.txt"
        # 计算 batch 级别的均值
        one_relative_L2_u = torch.sum(L2_relative_u).detach().cpu().item()
        one_num_u = u.shape[0]
        one_relative_L2_v = torch.sum(L2_relative_v).detach().cpu().item()
        one_num_v = v.shape[0]
        one_relative_L2_p = torch.sum(L2_relative_p).detach().cpu().item()
        one_num_p = p.shape[0]
        
        one_relative_L2_u /= one_num_u
        one_relative_L2_v /= one_num_v
        one_relative_L2_p /= one_num_p
        
        # 遍历整个 batch（每个样本都写入文件）
        for idx in range(flag_ID.shape[0]):  
            one_flag_ID = int(flag_ID[idx].detach().cpu().numpy().item())  # 取当前样本的 ID
            one_sample_L2_u = L2_relative_u[idx].detach().cpu().item()  # 取当前样本的 U 误差
            one_sample_L2_v = L2_relative_v[idx].detach().cpu().item()  # 取当前样本的 V 误差
            one_sample_L2_p = L2_relative_p[idx].detach().cpu().item()  # 取当前样本的 P 误差
        
            with open(file_path, "a") as f:
                f.write(f"{one_flag_ID} {one_relative_L2_u} {one_relative_L2_v} {one_relative_L2_p}\n")


    mean_relative_L2 /= num
    mean_relative_L2_u /= num_u
    mean_relative_L2_v /= num_v
    mean_relative_L2_p /= num_p
    
    # make the coordinates to numpy
    coor_x = test_coor_x[0].detach().cpu().numpy()
    coor_y = test_coor_y[0].detach().cpu().numpy()

    # color bar range
    max_color_u = np.amax(best_gt_u)
    min_color_u = np.amin(best_gt_u)
    max_color_worst_u = np.amax(worst_gt_u)
    min_color_worst_u = np.amin(worst_gt_u)
    # max_color_r_u = np.amax([np.amax(worst_gt_u-worst_f_u), np.amax(best_gt_u-best_f_u)])
    # min_color_r_u = np.amin([np.amin(worst_gt_u-worst_f_u), np.amin(best_gt_u-best_f_u)])
    max_color_r_u = np.amax([np.amax(np.abs(worst_gt_u - worst_f_u)), np.amax(np.abs(best_gt_u - best_f_u))])
    min_color_r_u = np.amin([np.amin(np.abs(worst_gt_u - worst_f_u)), np.amin(np.abs(best_gt_u - best_f_u))])

    max_color_v = np.amax(best_gt_v)
    min_color_v = np.amin(best_gt_v)
    max_color_worst_v = np.amax(worst_gt_v)
    min_color_worst_v = np.amin(worst_gt_v)
    max_color_r_v = np.amax([np.amax(np.abs(worst_gt_v - worst_f_v)), np.amax(np.abs(best_gt_v - best_f_v))])
    min_color_r_v = np.amin([np.amin(np.abs(worst_gt_v - worst_f_v)), np.amin(np.abs(best_gt_v - best_f_v))])

    max_color_p = np.amax(best_gt_p)
    min_color_p = np.amin(best_gt_p)
    max_color_worst_p = np.amax(worst_gt_p)
    min_color_worst_p = np.amin(worst_gt_p)
    max_color_r_p = np.amax([np.amax(np.abs(worst_gt_p - worst_f_p)), np.amax(np.abs(best_gt_p - best_f_p))])
    min_color_r_p = np.amin([np.amin(np.abs(worst_gt_p - worst_f_p)), np.amin(np.abs(best_gt_p - best_f_p))])


    if found_ID == int(target_id):
        max_found_u = np.amax(found_u)
        min_found_u = np.amin(found_u)
        max_found_r_u = np.amax(np.abs(found_u - found_u_pred))
        max_found_v = np.amax(found_v)
        min_found_v = np.amin(found_v)
        max_found_r_v = np.amax(np.abs(found_v - found_v_pred))
        max_found_p = np.amax(found_p)
        min_found_p = np.amin(found_p)
        max_found_r_p = np.amax(np.abs(found_p - found_p_pred))
    # make a plot
    SS = 1
    cm = plt.cm.get_cmap('RdYlBu_r')
    # cm = plt.cm.get_cmap('RdYlBu')
    # u #################################################################################################################
    fig, axes = plt.subplots(2, 3, figsize=(16, 8), dpi=400)  # 创建 2x3 的子图网格
    titles = [
        f'U Prediction (worst case, ID: {worst_flag_ID_u})',
        f'U Truth (worst case, ID: {worst_flag_ID_u})',
        f'U Absolute Error (worst case, ID: {worst_flag_ID_u})',
        f'U Prediction (best case, ID: {best_flag_ID_u})',
        f'U Truth (best case, ID: {best_flag_ID_u})',
        f'U Absolute Error (best case, ID: {best_flag_ID_u})'
    ]

    datasets = [
        worst_f_u, worst_gt_u, np.abs(worst_f_u - worst_gt_u),
        best_f_u, best_gt_u, np.abs(best_f_u - best_gt_u)
    ]
    color_ranges = [
        (min_color_worst_u, max_color_worst_u),
        (min_color_worst_u, max_color_worst_u),
        (0, max_color_r_u),
        (min_color_u, max_color_u),
        (min_color_u, max_color_u),
        (0, max_color_r_u)
    ]
    
    from mpl_toolkits.axes_grid1 import make_axes_locatable
    
    for i, ax in enumerate(axes.flatten()):
        data = datasets[i]
        vmin, vmax = color_ranges[i]
        scatter = ax.scatter(coor_x, coor_y, c=data, cmap=cm, vmin=vmin, vmax=vmax, marker='o', s=SS)
    
        # 添加自定义的颜色条
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.05)  # 控制颜色条宽度和间距
        fig.colorbar(scatter, cax=cax)
    
        ax.set_title(titles[i], fontsize=15)
        ax.set_aspect('equal')
        # 关闭子图框线
        ax.axis('off')  # 关闭坐标轴（包括框线）
    fig.subplots_adjust(hspace=0.1, wspace=0.3, top=0.8, bottom=0.2)  # 调整上下间距，并向中间收拢
    plt.savefig('u_plots.png')  # 保存 u 图像
    plt.show()
    
    # v #################################################################################################################
    fig, axes = plt.subplots(2, 3, figsize=(16, 8), dpi=400)  # 创建 2x3 的子图网格
    titles = [
        f'V Prediction (worst case, ID: {worst_flag_ID_v})',
        f'V Truth (worst case, ID: {worst_flag_ID_v})',
        f'V Absolute Error (worst case, ID: {worst_flag_ID_v})',
        f'V Prediction (best case, ID: {best_flag_ID_v})',
        f'V Truth (best case, ID: {best_flag_ID_v})',
        f'V Absolute Error (best case, ID: {best_flag_ID_v})'
    ]
    datasets = [
        worst_f_v, worst_gt_v, np.abs(worst_f_v - worst_gt_v),
        best_f_v, best_gt_v, np.abs(best_f_v - best_gt_v)
    ]
    color_ranges = [
        (min_color_worst_v, max_color_worst_v),
        (min_color_worst_v, max_color_worst_v),
        (0, max_color_r_v),
        (min_color_v, max_color_v),
        (min_color_v, max_color_v),
        (0, max_color_r_v)
    ]
    
    for i, ax in enumerate(axes.flatten()):
        data = datasets[i]
        vmin, vmax = color_ranges[i]
        scatter = ax.scatter(coor_x, coor_y, c=data, cmap=cm, vmin=vmin, vmax=vmax, marker='o', s=SS)
    
        # 添加自定义的颜色条
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.05)
        fig.colorbar(scatter, cax=cax)
    
        ax.set_title(titles[i], fontsize=15)
        ax.set_aspect('equal')
        # 关闭子图框线
        ax.axis('off')  # 关闭坐标轴（包括框线）
    fig.subplots_adjust(hspace=0.1, wspace=0.3, top=0.8, bottom=0.2)  # 调整上下间距，并向中间收拢
    plt.savefig('v_plots.png')  # 保存 u 图像
    plt.show()
    
    # p #################################################################################################################
    fig, axes = plt.subplots(2, 3, figsize=(16, 8), dpi=400)  # 创建 2x3 的子图网格
    titles = [
        f'P Prediction (worst case, ID: {worst_flag_ID_p})',
        f'P Truth (worst case, ID: {worst_flag_ID_p})',
        f'P Absolute Error (worst case, ID: {worst_flag_ID_p})',
        f'P Prediction (best case, ID: {best_flag_ID_p})',
        f'P Truth (best case, ID: {best_flag_ID_p})',
        f'P Absolute Error (best case, ID: {best_flag_ID_p})'
    ]
    datasets = [
        worst_f_p, worst_gt_p, np.abs(worst_f_p - worst_gt_p),
        best_f_p, best_gt_p, np.abs(best_f_p - best_gt_p)
    ]
    color_ranges = [
        (min_color_worst_p, max_color_worst_p),
        (min_color_worst_p, max_color_worst_p),
        (0, max_color_r_p),
        (min_color_p, max_color_p),
        (min_color_p, max_color_p),
        (0, max_color_r_p)
    ]
    
    for i, ax in enumerate(axes.flatten()):
        data = datasets[i]
        vmin, vmax = color_ranges[i]
        scatter = ax.scatter(coor_x, coor_y, c=data, cmap=cm, vmin=vmin, vmax=vmax, marker='o', s=SS)
    
        # 添加自定义的颜色条
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.05)
        fig.colorbar(scatter, cax=cax)
    
        ax.set_title(titles[i], fontsize=15)
        ax.set_aspect('equal')
        # 关闭子图框线
        ax.axis('off')  # 关闭坐标轴（包括框线）
    fig.subplots_adjust(hspace=0.1, wspace=0.3, top=0.8, bottom=0.2)  # 调整上下间距，并向中间收拢
    plt.savefig('p_plots.png')  # 保存 u 图像
    plt.show()

    # 目标ID #################################################################################################################

    if found_ID == int(target_id):
        fig, axes = plt.subplots(3, 3, figsize=(16, 8), dpi=400)  # 创建 2x3 的子图网格
        titles = [
            f'U Prediction (target_id: {target_id})',
            f'U Truth (target_id: {target_id})',
            f'U Absolute Error (target_id: {target_id})',
            f'V Prediction (target_id: {target_id})',
            f'V Truth (target_id: {target_id})',
            f'V Absolute Error (target_id: {target_id})',
            f'P Prediction (target_id: {target_id})',
            f'P Truth (target_id: {target_id})',
            f'P Absolute Error (target_id: {target_id})',
        ]
        datasets = [
            found_u_pred, found_u, np.abs(found_u - found_u_pred),
            found_v_pred, found_v, np.abs(found_v - found_v_pred),
            found_p_pred, found_p, np.abs(found_p - found_p_pred)
        ]
        
        color_ranges = [
            (min_found_u, max_found_u),
            (min_found_u, max_found_u),
            (0, max_found_r_u),
            (min_found_v, max_found_v),
            (min_found_v, max_found_v),
            (0, max_found_r_v),
            (min_found_p, max_found_p),
            (min_found_p, max_found_p),
            (0, max_found_r_p)
        ]  
        for i, ax in enumerate(axes.flatten()):
            data = datasets[i]
            vmin, vmax = color_ranges[i]
            scatter = ax.scatter(coor_x, coor_y, c=data, cmap=cm, vmin=vmin, vmax=vmax, marker='o', s=SS)
        
            # 添加自定义的颜色条
            divider = make_axes_locatable(ax)
            cax = divider.append_axes("right", size="5%", pad=0.05)
            fig.colorbar(scatter, cax=cax)
        
            ax.set_title(titles[i], fontsize=15)
            ax.set_aspect('equal')
            # 关闭子图框线
            ax.axis('off')  # 关闭坐标轴（包括框线）
        fig.subplots_adjust(hspace=0.3, wspace=0.3, top=0.95, bottom=0.1)  # 调整上下间距，并向中间收拢
        plt.savefig('id_plots.png')  # 保存 u 图像
        plt.show()

    #plt.savefig(r'sample_{}_{}.png'.format(args.model_u, args.data))
    return mean_relative_L2,mean_relative_L2_u,mean_relative_L2_v,mean_relative_L2_p

In [ ]:
# 定义坐标噪声添加函数
def add_noise(u, noise_level, noise_type='gaussian'):
    """
    为坐标添加噪声
    """
    if noise_level <= 0:
        return u
    
    if noise_type == 'gaussian':
        std = torch.std(u)
        noise = torch.randn_like(u) * std * noise_level
        return u + noise
    elif noise_type == 'uniform':
        std = torch.std(u)
        noise = (torch.rand_like(u) * 2 - 1) * std * noise_level
        return u + noise
    else:
        raise ValueError("noise_type must be 'gaussian' or 'uniform'")

In [ ]:

# define the training function
def train(args, config, model, device, loaders, coors, flag_BCxy, flag_BCy, flag_BC_load, params):

    # print training configuration
    print('training configuration')
    print('batchsize:', config['train']['batchsize'])
    print('coordinate sampling frequency:', config['train']['coor_sampling_freq'])
    print('learning rate:', config['train']['base_lr'])
    print('BC weight', config['train']['bc_weight'])
    print('PDE weight', config['train']['pde_weight'])

    # get train and test loader
    train_loader, val_loader, test_loader = loaders

    # define model training configuration 定义模型训练配置
    pbar = range(config['train']['epochs'])
    pbar = tqdm(pbar, dynamic_ncols=True, smoothing=0.1)

    # extract coors
    xy_BC_coors = coors[np.where(flag_BCxy==1)[0],:]
    y_BC_coors = coors[np.where(flag_BCy==1)[0],:]
    load_coors = coors[np.where(flag_BC_load==1)[0],:]
    load_BC_coors = coors[np.where(flag_BCxy+flag_BCy+flag_BC_load==0)[0],:]
    pde_coors = coors[np.where(flag_BCxy+flag_BCy+flag_BC_load==0)[0],:]
    num_pde_nodes = pde_coors.shape[0]

    # Move the data to device
    xy_BC_coors = xy_BC_coors.float().to(device)
    y_BC_coors = y_BC_coors.float().to(device)
    load_coors = load_coors.float().to(device)
    load_BC_coors = load_BC_coors.float().to(device)
    coors = coors.float().to(device)
    print('Number of PDE points:', num_pde_nodes)

    # define optimizer and loss
    mse = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=config['train']['base_lr'])


    #用于评估和存储更新后的模型参数的可视频率
    vf = config['train']['visual_freq']
    
    # 使用 DataParallel 来启用多 GPU 训练
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model = model.to(device)
    # initialize recored loss values
    avg_loss1 = np.inf
    avg_loss2 = np.inf
    avg_loss21 = np.inf
    avg_loss3 = np.inf
    avg_loss4 = np.inf
    avg_loss5 = np.inf
    avg_loss6 = np.inf
    avg_loss7 = np.inf
    avg_loss8 = np.inf
    avg_loss9 = np.inf

    try:
        model.load_state_dict(torch.load(f'best_model_{args.data}.pth', map_location=device))  
        
        print(" Models loaded successfully.")
    except FileNotFoundError:
        print("No pre-trained model found.")
    except Exception as e:
        print(f"Error while loading models: {e}")

    # Initialize relative error record
    relative_errors = []
    actual_epochs = []

    # start the training
    if args.phase == 'train':
        min_val_err = np.inf
        min_val_err_u = np.inf
        min_val_err_v = np.inf
        min_val_err_p = np.inf
        
        for e in pbar:
          
            # show the performance improvement
            if e % vf == 0:
                model.eval()
                model.eval()
                model.eval()
                err_u, err_v, err_p, err = val(model, val_loader, coors, device)
                
                # relative_errors.append(err)  # 记录相对误差
                relative_errors.append((err_u, err_v, err_p, err))  # 添加一个元组
                actual_epochs.append(e)
                
                print('Best L2 relative error:', err)
                print('Best u  relative error:', err_u)
                print('Best v  relative error:', err_v)
                print('Best p  relative error:', err_p)
                print('u-direction  loss', avg_loss1)
                print('u0-direction loss:', avg_loss2)
                print('ru-direction loss:', avg_loss3)
                print('v-direction  loss', avg_loss4)
                print('v0-direction loss:', avg_loss5)
                print('rv-direction loss:', avg_loss6)
                print('p-direction  loss', avg_loss7)
                print('p0-direction loss:', avg_loss8)
                print('rp-direction loss:', avg_loss9)

                if err < min_val_err_u:
                    torch.save(model.state_dict(), f'best_model_{args.data}.pth')  
                    min_val_err_u = err_u

                
                # update recored loss values
                avg_loss1 = 0
                avg_loss2 = 0
                avg_loss3 = 0
                avg_loss4 = 0
                avg_loss5 = 0
                avg_loss6 = 0
                avg_loss7 = 0
                avg_loss8 = 0
                avg_loss9 = 0

            # train one epoch
            model.train()
            
            for (par_u, par_v, par_p, u, v, p) in train_loader:

                for _ in range(config['train']['coor_sampling_freq']):

                    # compute the sampling probability of each collocation point
                    batchsize = u.shape[0] 
                    # prepare the data
                    par_u = par_u.float().to(device)
                    par_v = par_v.float().to(device)
                    par_p = par_p.float().to(device)
                    xy_BC_coors_r = xy_BC_coors.unsqueeze(0).repeat(batchsize,1,1)
                    # y_BC_coors_r = y_BC_coors.unsqueeze(0).repeat(batchsize,1,1)
                    load_coors_r = load_coors.unsqueeze(0).repeat(batchsize,1,1)
                    load_BC_coors_r = load_BC_coors.unsqueeze(0).repeat(batchsize,1,1)
                    # load_BC_coors_r = add_coordinate_noise(load_BC_coors_r, args.noise_level, noise_type)

                    # u #########################################################################
                    u_pred, v_pred, p_pred = model(load_BC_coors_r[:, :, 0], load_BC_coors_r[:, :, 1], par_u, par_v, par_p)
                    u_gt1 = u[:, np.where(flag_BCxy+flag_BCy+flag_BC_load==0)[0]].float().to(device)
                    v_gt1 = v[:, np.where(flag_BCxy+flag_BCy+flag_BC_load==0)[0]].float().to(device)
                    p_gt1 = p[:, np.where(flag_BCxy+flag_BCy+flag_BC_load==0)[0]].float().to(device)
                    if hasattr(args, 'noise_level') and args.noise_level > 0:
                        noise_type = getattr(args, 'noise_type', 'gaussian')
                        u_gt = add_noise(u_gt1, args.noise_level, noise_type)
                        v_gt = add_noise(v_gt1, args.noise_level, noise_type)
                        p_gt = add_noise(p_gt1, args.noise_level, noise_type)

                    u_BCxy_pred, v_BCxy_pred, p_BCxy_pred = model(xy_BC_coors_r[:, :, 0], xy_BC_coors_r[:, :, 1], par_u, par_v, par_p)
                    u_bc_gt = u[:, np.where(flag_BCxy == 1)[0]].float().to(device)
                    v_bc_gt = v[:, np.where(flag_BCxy == 1)[0]].float().to(device)
                    p_bc_gt = p[:, np.where(flag_BCxy == 1)[0]].float().to(device)

                    u_load_pred, v_load_pred, p_load_pred = model(load_coors_r[:, :, 0], load_coors_r[:, :, 1], par_u, par_v, par_p)
                    u_load_gt = u[:, np.where(flag_BC_load == 1)[0]].float().to(device)
                    v_load_gt = v[:, np.where(flag_BC_load == 1)[0]].float().to(device)
                    p_load_gt = p[:, np.where(flag_BC_load == 1)[0]].float().to(device)

                    # compute the loss
                    bc_loss_u1 = mse(u_pred, u_gt)
                    bc_loss_u2 = mse(u_BCxy_pred, u_bc_gt)
                    bc_loss_u3 = mse(u_load_pred, u_load_gt)
                    total_loss_u = bc_loss_u1*20 + bc_loss_u2*10 + bc_loss_u3

                    bc_loss_v1 = mse(v_pred, v_gt)
                    bc_loss_v2 = mse(v_BCxy_pred, v_bc_gt)
                    bc_loss_v3 = mse(v_load_pred, v_load_gt)
                    total_loss_v = bc_loss_v1*20 + bc_loss_v2*10 + bc_loss_v3

                    # compute the loss
                    bc_loss_p1 = mse(p_pred, p_gt)
                    bc_loss_p2 = mse(p_BCxy_pred, p_bc_gt)
                    bc_loss_p3 = mse(p_load_pred, p_load_gt)
                    total_loss_p = bc_loss_p1*20 + bc_loss_p2*10 + bc_loss_p3

                    total_loss= total_loss_u + total_loss_v + total_loss_p
                    # store the loss
                    avg_loss1 += bc_loss_u1.detach().cpu().item()
                    avg_loss2 += bc_loss_u2.detach().cpu().item()
                    avg_loss3 += bc_loss_u3.detach().cpu().item()
                    
                    avg_loss4 += bc_loss_v1.detach().cpu().item()
                    avg_loss5 += bc_loss_v2.detach().cpu().item()
                    avg_loss6 += bc_loss_v3.detach().cpu().item()
                    
                    avg_loss7 += bc_loss_p1.detach().cpu().item()
                    avg_loss8 += bc_loss_p2.detach().cpu().item()
                    avg_loss9 += bc_loss_p3.detach().cpu().item()
                    
                    # update parameter
                    optimizer.zero_grad()
                    total_loss.backward()
                    optimizer.step()




    # 定义文件路径变量
    file_path = 'relative_errors_{}.txt'.format(args.data)
        
    # 保存相对误差到文件
    if args.phase == 'train':
        if actual_epochs:
            save_relative_errors(relative_errors, actual_epochs, file_path, config, args)
            print("Relative errors have been saved to '{}'.".format(file_path))
            
     # 示例：读取并绘制相对误差图，文件路径是相对路径
    plot_relative_errors_from_file(file_path)

   # # 示例：读取并绘制相对误差图，文件路径是相对路径
    # plot_relative_errors_from_file('relative_errors_{}_{}.txt'.format(args.data, args.model))
    
    # final test
    model.load_state_dict(torch.load(f'best_model_{args.data}.pth', map_location=device))   
    model.eval()

    err,err_u,err_v,err_p = test(model, test_loader, coors, device, args)

    print('Best L2 relative error on test loader:', err)
    print('Best L2 relative error_u on test loader:', err_u)
    print('Best L2 relative error_v on test loader:', err_v)
    print('Best L2 relative error_p on test loader:', err_p)
    file_path = "relative_L2.txt"
    with open(file_path, "a") as f:
        f.write(f"TM {err_u} {err_v} {err_p}\n")

In [ ]:
with open(r'ns.yaml', 'r') as stream:
    config = yaml.load(stream, yaml.FullLoader)
    print(config)

In [ ]:
import argparse

args = argparse.Namespace(
    phase='train', 
    target_id='203', 
    data='ns', 
    model='DeepMONet', 
    noise_level=0.01,  # 坐标噪声水平
    noise_type='gaussian' , # 噪声类型
)

if args.model == 'DeepMONet':
    model = DeepMONet(config)

In [ ]:
train(args, config, model, device, (train_loader, val_loader, test_loader), coors, flag_BCxy, flag_BCy, flag_load, [dens, mu])